# Create Gairdner Awards (PRIZE PATTERN)

Canada Gairdner Award laureates from the official Gairdner Sitefinity OData API that backs `https://www.gairdner.org/winners`.

**Awarding body:** Gairdner Foundation - F4320313415

**Prerequisites:**
- Run `scripts/local/gairdner_to_s3.py` first.
- Admin must upload the generated parquet because contractor access has no S3 credentials.

**Data source:** `https://www.gairdner.org/api/default/awardwinners?$expand=RelatedAward,RelatedWinner`

**S3:** `s3a://openalex-ingest/awards/gairdner/gairdner_awards.parquet`

**Funder details:**
- funder_id: `4320313415`
- display_name: `Gairdner Foundation`
- ror_id: `https://ror.org/003ex4g35`
- doi: `10.13039/100012267`

Schema notes:
- One row per `(award x laureate)` from the official `awardwinners` API.
- `lead_investigator` is the laureate; `funder_scheme` is the Gairdner award program.
- `amount` is NULL and `currency` is CAD. Gairdner prizes are monetary, but the public API does not expose a reliable historical per-laureate amount, older values changed over time, and some awards can be shared. Step 6.7 amount coverage is explicitly waived for this prize-pattern ingest.
- Source smoke test on 2026-05-18 returned 443 API rows.
- Priority 62 is used because priorities 55-61 are reserved for in-flight PRs #85-#88 plus USGS, NIST, and IES.


## Step 1: Create staging table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gairdner_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/gairdner/gairdner_awards.parquet`;


## Step 1.5: Inspect raw data before transform


In [ ]:
%sql
SELECT COUNT(*) AS raw_rows FROM openalex.awards.gairdner_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.gairdner_raw;


In [ ]:
%sql
SELECT * FROM openalex.awards.gairdner_raw LIMIT 10;


In [ ]:
%sql
-- Required money-field scan. The official API does not expose a reliable
-- historical per-laureate prize amount, so amount is mapped to NULL below.
SELECT column_name
FROM (DESCRIBE openalex.awards.gairdner_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
-- Required currency-field scan. Currency is CAD by awarding-body context,
-- but amount remains NULL because historical per-laureate values are not in source.
SELECT column_name
FROM (DESCRIBE openalex.awards.gairdner_raw)
WHERE LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
SELECT
    award_name,
    COUNT(*) AS rows,
    MIN(TRY_CAST(year AS INT)) AS min_year,
    MAX(TRY_CAST(year AS INT)) AS max_year,
    COUNT(citation) AS rows_with_citation,
    COUNT(laureate_position_title) AS rows_with_position_title
FROM openalex.awards.gairdner_raw
GROUP BY award_name
ORDER BY rows DESC;


## Step 1.6: Funder existence fail-fast


In [ ]:
%sql
-- Must return exactly 1 row. If 0 rows, STOP and do not run the transform.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320313415;


## Step 2: Transform to award schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gairdner_awards
USING delta
AS
WITH gairdner_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320313415
),
normalized AS (
    SELECT
        g.*,
        TRY_CAST(g.year AS INT) AS award_year
    FROM openalex.awards.gairdner_raw g
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':gairdner:', LOWER(TRIM(g.funder_award_id))))) % 9000000000 AS id,
    CONCAT(g.award_name, ' ', CAST(g.award_year AS STRING), ' - ', g.laureate_name) AS display_name,
    CASE
        WHEN TRY_CAST(g.declined AS BOOLEAN)
             AND COALESCE(NULLIF(g.citation, ''), NULLIF(g.description, ''), NULLIF(g.award_summary, '')) IS NOT NULL
            THEN CONCAT('Declined the prize. ', COALESCE(NULLIF(g.citation, ''), NULLIF(g.description, ''), NULLIF(g.award_summary, '')))
        WHEN TRY_CAST(g.declined AS BOOLEAN) THEN 'Declined the prize.'
        ELSE COALESCE(NULLIF(g.citation, ''), NULLIF(g.description, ''), NULLIF(g.award_summary, ''))
    END AS description,
    f.funder_id,
    g.funder_award_id AS funder_award_id,
    CAST(NULL AS DOUBLE) AS amount,         -- prize-pattern waiver: historical per-laureate amount not in source API
    'CAD' AS currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) AS id,
        f.display_name, f.ror_id, f.doi
    ) AS funder,
    'prize' AS funding_type,
    g.award_name AS funder_scheme,
    'gairdner_sitefinity' AS provenance,
    TRY_TO_DATE(CONCAT(CAST(g.award_year AS STRING), '-01-01'), 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(CONCAT(CAST(g.award_year AS STRING), '-12-31'), 'yyyy-MM-dd') AS end_date,
    g.award_year AS start_year,
    g.award_year AS end_year,
    struct(
        NULLIF(g.laureate_given_name, '') AS given_name,
        NULLIF(g.laureate_family_name, '') AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            NULLIF(g.laureate_position_title, '') AS name,
            CAST(NULL AS STRING) AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    g.laureate_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':gairdner:', LOWER(TRIM(g.funder_award_id))))) % 9000000000) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM normalized g
CROSS JOIN gairdner_funder f
WHERE g.funder_award_id IS NOT NULL
  AND g.laureate_name IS NOT NULL
  AND g.award_name IS NOT NULL
  AND g.award_year IS NOT NULL;


## Step 3: Insert into openalex_awards_raw at priority 62


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'gairdner_sitefinity' AND priority = 62;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT id, display_name, description, funder_id, funder_award_id,
       amount, currency, funder, funding_type, funder_scheme, provenance,
       start_date, end_date, start_year, end_year,
       lead_investigator, co_lead_investigator, investigators,
       landing_page_url, doi, works_api_url,
       created_date, updated_date,
       62 AS priority
FROM openalex.awards.gairdner_awards;


## Verification


In [ ]:
%sql
-- Step 6.7 amount/currency check - WAIVED for Gairdner because the source API
-- lacks reliable historical per-laureate prize amounts. Expect pct_amount = 0
-- and currencies = [CAD].
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    collect_set(currency) AS currencies,
    COUNT(DISTINCT funder_scheme) AS distinct_award_programs
FROM openalex.awards.gairdner_awards;


In [ ]:
%sql
SELECT
    id,
    SUBSTRING(display_name, 1, 100) AS display_name,
    funder_scheme,
    start_year,
    lead_investigator.given_name,
    lead_investigator.family_name,
    lead_investigator.affiliation.name AS affiliation,
    landing_page_url
FROM openalex.awards.gairdner_awards
ORDER BY start_year DESC, funder_scheme, lead_investigator.family_name
LIMIT 20;


In [ ]:
%sql
SELECT id, COUNT(*) AS dupes
FROM openalex.awards.gairdner_awards
GROUP BY id
HAVING COUNT(*) > 1;


In [ ]:
%sql
SELECT
    funder_scheme,
    COUNT(*) AS rows,
    MIN(start_year) AS min_year,
    MAX(start_year) AS max_year
FROM openalex.awards.gairdner_awards
GROUP BY funder_scheme
ORDER BY rows DESC;


In [ ]:
%sql
-- §6.3 Data completeness (runbook §6.3 form, adapted for prize pattern).
-- has_amount is expected to be 0 — amount is waived for Gairdner per Step 6.7 note.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    COUNT(lead_investigator.given_name) AS has_given_name,
    COUNT(lead_investigator.family_name) AS has_family_name,
    COUNT(lead_investigator.affiliation.name) AS has_affiliation,
    COUNT(funder_award_id) AS has_funder_award_id,
    COUNT(landing_page_url) AS has_landing_page_url,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_dates
FROM openalex.awards.gairdner_awards;

In [ ]:
%sql
SELECT funder_id, funder.display_name, COUNT(*) AS rows
FROM openalex.awards.gairdner_awards
GROUP BY funder_id, funder.display_name;


In [ ]:
%sql
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.gairdner_awards
GROUP BY start_year
ORDER BY start_year DESC;


In [ ]:
%sql
SELECT provenance, priority, COUNT(*) AS inserted_rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'gairdner_sitefinity' AND priority = 62
GROUP BY provenance, priority;
